# 1. Import of input data

In [ ]:
import nbimporter
import sys
import os
from pathlib import Path

In [ ]:
#Import libraries
try:
    current_dir = RUN_DIR
    print(f"📂 CombinedHeatPower_CombinedCycle.ipynb executed by RUN.ipynb — using directory: {current_dir}")
except NameError:

    current_dir = Path().resolve()
    print(f"📁 CombinedHeatPower_CombinedCycle.ipynb executed standalone — using directory: {current_dir}")

Libraries_path = (current_dir / "Libraries.ipynb").resolve()

# If not found, try in the parent levels
if not Libraries_path.exists():
    Libraries_path = (current_dir.parent / "Libraries.ipynb").resolve()

# Run the file
if Libraries_path.exists():
    print(f"📘 Importing libraries from: {Libraries_path}")
    get_ipython().run_line_magic('run', f'"{Libraries_path}"')
else:
    print("⚠️ Libraries.ipynb not found! Some functions might not be available.")

In [ ]:
energysystem = pd.read_excel(energysystem_path, sheet_name = 'CHP-Combined Cycle')

# 2. Input parameters

In [ ]:
Power = energysystem['Thermal capacity [kW]']
EfficiencyW = energysystem['Electrical efficiency based on LHV (between 0 and 1)']
EfficiencyQ = energysystem['Heat efficiency based on LHV (between 0 and 1)']
ElectricEnergyproduced = energysystem['Electric energy produced in a year (exported and consumed) [kWh /year]']
ThermalEnergyproduced_primary = energysystem['Thermal energy produced used [kWh /year]']
Qused = 1
SupplyTemperature = energysystem['Supply temperature [°C]']
ReturnTemperature = energysystem['Return temperature [°C]']
EnergyInputSourceGasBlend = energysystem['Energy input from gas-blend [kWh/year]']
EnergyInputSourceBiogas = energysystem['Energy input from biogas [kWh/year]']
EnergyInputSourceHydrogen = energysystem['Energy input from hydrogen [kWh/year]']

EquivalentHours = ThermalEnergyproduced_primary / Power

In [ ]:
# Natural gas
LHV = 36.8  # In MJ per Nm3 - Table 3.6 06_V_Natural Gas 3_3 Fuel Data
HHV = 40.4  # In MJ per Nm3 - Table 3.6 06_V_Natural Gas 3_3 Fuel Data

# Biomethane
LHVBiom = 35    # MJ/m3
HHVBiom = 38    # MJ/m3

# Hydrogen
LHVH2 = 120     # MJ/kg
HHVH2 = 142     # MJ/kg 

In [ ]:
Lifespan = np.where(
    Power * (EfficiencyW / EfficiencyQ)  < 25,
    40000 / EquivalentHours,
    100000 / EquivalentHours
)

In [ ]:
# ADDITIONAL CALCULATIONS
Power = np.array(Power)
Lifespan = np.array(Lifespan)
EfficiencyW = np.array(EfficiencyW)
EfficiencyQ = np.array(EfficiencyQ)
EfficiencyTot = EfficiencyW + EfficiencyQ
EnergyProducedLHV = (np.asarray(ThermalEnergyproduced_primary, dtype=float) * RSP) / EfficiencyQ   # fuel over RSP
EnergyProducedHHV = EnergyProducedLHV * (HHV/LHV)
TotW = EnergyProducedLHV * EfficiencyW
TotQ = EnergyProducedLHV * EfficiencyQ * Qused

In [ ]:
# MULTIFUNCTIONALITY AND ALLOCATION OF OUTPUTS
# ------------------------------------------------------------------
CombinedCycle_Q_RSP = np.asarray(ThermalEnergyproduced_primary, dtype=float) * RSP          # kWh_th
CombinedCycle_W_RSP = CombinedCycle_Q_RSP * (np.asarray(EfficiencyW, dtype=float)
                           / np.asarray(EfficiencyQ, dtype=float))                   # kWh_el

# Consistency check against the electricity declared in the input sheet [kWh/year]
_W_input_RSP = np.asarray(ElectricEnergyproduced, dtype=float) * RSP
_mismatch = np.abs(_W_input_RSP - CombinedCycle_W_RSP) > 0.05 * np.maximum(CombinedCycle_W_RSP, 1e-9)
if np.any(_mismatch & (_W_input_RSP > 0)):
    print("⚠️ CombinedCycle: declared electricity differs by >5% from Q·ηW/ηQ; "
          "allocation uses the efficiency-based value.")

# Rows without supply/return temperatures (both 0 °C): checked BEFORE converting to K
CombinedCycle_NoTemperatures = ((np.asarray(SupplyTemperature, dtype=float) == 0)
                       & (np.asarray(ReturnTemperature, dtype=float) == 0))

if Multifunctionality == 'Default':
    # Power-loss method: electricity lost because heat is extracted, compared with
    # condensing operation at 55 % electrical efficiency (LHV)
    CombinedCycle_F_RSP = CombinedCycle_Q_RSP / np.asarray(EfficiencyQ, dtype=float)            # kWh fuel (LHV)
    DeltaElProduced = CombinedCycle_F_RSP * 0.55 - CombinedCycle_W_RSP
    with np.errstate(divide='ignore', invalid='ignore'):
        AFQ = DeltaElProduced / (CombinedCycle_W_RSP + DeltaElProduced)

    AFQ_CombinedCycle = np.nan_to_num(np.asarray(AFQ, dtype=float).reshape(-1))
    AFW_CombinedCycle = 1.0 - AFQ_CombinedCycle

else:
    # Power bonus method (EN 15316-4-5): the exported electricity is credited with the
    # primary energy it displaces; the heat takes the remainder of the fuel.
    # Share of the fuel (primary energy) attributed to electricity:
    #   AFW = W * f_el / (F * f_fuel),   AFQ = 1 - AFW
    f_fuel = 1.05   # primary-energy factor of the fuel
    f_el = 2.42     # primary-energy factor of exported electricity
    CombinedCycle_F_LHV_RSP = CombinedCycle_Q_RSP / np.asarray(EfficiencyQ, dtype=float)   # kWh fuel (LHV)
    AFW_raw = np.asarray((CombinedCycle_W_RSP * f_el) / (CombinedCycle_F_LHV_RSP * f_fuel), dtype=float).reshape(-1)
    if np.any(AFW_raw > 1):
        print("⚠️ CombinedCycle: power bonus exceeds the fuel (AFW > 1); AFW capped at 1, heat burden = 0:",
              np.round(AFW_raw, 3))
    AFW_CombinedCycle = np.clip(AFW_raw, 0.0, 1.0)
    AFQ_CombinedCycle = 1.0 - AFW_CombinedCycle

AFQ_CombinedCycle = np.where(CombinedCycle_NoTemperatures, 0.0, AFQ_CombinedCycle)
AFW_CombinedCycle = np.where(CombinedCycle_NoTemperatures, 0.0, AFW_CombinedCycle)

In [ ]:
PowerEle = (Power / EfficiencyQ) * EfficiencyW 
RefPowerEle = 430000 #kW
RefMass = 2.79E+07
mass = PowerEle * (RefMass / RefPowerEle)

# 3. Life cycle inventories

## 3.1 Module A1-3

In [ ]:
CombinedCycle_aluminiumCastalloy = 0.0105 * mass * AFQ_CombinedCycle
CombinedCycle_aluminiumWroughtalloy = 0.0222 * mass * AFQ_CombinedCycle
CombinedCycle_ceramictile = 0.00003 * mass * AFQ_CombinedCycle
CombinedCycle_chromium = 0.00007 * mass * AFQ_CombinedCycle
CombinedCycle_cobalt = 0.00005 * mass * AFQ_CombinedCycle
CombinedCycle_concreteAT = 2127 * (PowerEle / RefPowerEle) * AFQ_CombinedCycle
CombinedCycle_concreteCH = 3872 * (PowerEle / RefPowerEle) * AFQ_CombinedCycle
CombinedCycle_copper = 0.0327 * mass * AFQ_CombinedCycle
CombinedCycle_diesel = 3441860 * (PowerEle / RefPowerEle) * AFQ_CombinedCycle
CombinedCycle_electricityA13 = 3020000 * (PowerEle / RefPowerEle) * AFQ_CombinedCycle
CombinedCycle_heat = 140600000 * (PowerEle / RefPowerEle) * AFQ_CombinedCycle
CombinedCycle_nickel = 0.0047 * mass * AFQ_CombinedCycle
CombinedCycle_PEHD = 0.031 * mass * AFQ_CombinedCycle
CombinedCycle_reinforcingsteel = 0.6542 * mass * AFQ_CombinedCycle
CombinedCycle_chromiumsteel = 0.0645 * mass * AFQ_CombinedCycle
CombinedCycle_stonewool = 0.0 * mass * AFQ_CombinedCycle

## 3.2 Module A4

In [ ]:
CombinedCycle_Truk_16_32 = (((500 + 200)*(mass)/1000)) * AFQ_CombinedCycle
CombinedCycle_Light_Truk = ((50*(mass)/1000)) * AFQ_CombinedCycle

## 3.3 Module B2

In [ ]:
CombinedCycle_LightTruk_B2 = ((50 * (Lifespan/2) * 0.01 * (mass/1000))) * AFQ_CombinedCycle

## 3.4 Module B4

In [ ]:
CombinedCycleNumberReplacements = (np.ceil(np.divide(ReqSL, Lifespan) - 1))
SFCombinedCycle = (RSP / ReqSL)  # Scaling factor

## 3.5 Module B6

In [ ]:
#Repartition parameters for energy source input
GasBlend = EnergyInputSourceGasBlend / (EnergyInputSourceGasBlend + EnergyInputSourceBiogas + EnergyInputSourceHydrogen)
Biogas = EnergyInputSourceBiogas / (EnergyInputSourceGasBlend + EnergyInputSourceBiogas + EnergyInputSourceHydrogen)
Hydrogen = EnergyInputSourceHydrogen / (EnergyInputSourceGasBlend + EnergyInputSourceBiogas + EnergyInputSourceHydrogen)


In [ ]:
# ------------------------------------------------------------------
# B6 - Combined Cycle
# Fuel balances and direct emissions are expressed on an LHV basis.
# ------------------------------------------------------------------

# Useful thermal output over the reference study period [MJ heat]
combined_cycle_heat_output_MJ = (
    np.asarray(ThermalEnergyproduced_primary, dtype=float)
    * RSP
    * 3.6
)

# Total CHP fuel input required for the useful heat [MJ fuel, LHV]
combined_cycle_total_fuel_input_MJ = (
    combined_cycle_heat_output_MJ
    / EfficiencyQ
)

# Fuel input allocated to the thermal co-product [MJ fuel, LHV]
combined_cycle_fuel_energy_MJ = (
    combined_cycle_total_fuel_input_MJ
    * AFQ_CombinedCycle
)

# ------------------------------------------------------------------
# Fuel inputs by origin - direct-emission basis [MJin, LHV]
# ------------------------------------------------------------------

# Gas blend: natural gas share
CombinedCycle_GasBlend_NG_MJin = (
    combined_cycle_fuel_energy_MJ
    * GasBlend
    * A
)

# Gas blend: biomethane share
CombinedCycle_GasBlend_Biomethane_MJin = (
    combined_cycle_fuel_energy_MJ
    * GasBlend
    * B
)

# Gas blend: hydrogen share
CombinedCycle_GasBlend_Hydrogen_MJin = (
    combined_cycle_fuel_energy_MJ
    * GasBlend
    * C
)

# Standalone biomethane input.
# The legacy variable "Biogas" is treated as biomethane.
CombinedCycle_Biomethane_MJin = (
    combined_cycle_fuel_energy_MJ
    * Biogas
)

# Standalone hydrogen input
CombinedCycle_Hydrogen_MJin = (
    combined_cycle_fuel_energy_MJ
    * Hydrogen
)

# ------------------------------------------------------------------
# Inputs to ecoinvent
# ------------------------------------------------------------------

# Gas blend - natural gas [m3]
CombinedCycle_GasBlend_NG_Input = (
    CombinedCycle_GasBlend_NG_MJin / LHV
)

# Gas blend - biomethane [m3]
CombinedCycle_GasBlend_Biomethane_Input = (
    CombinedCycle_GasBlend_Biomethane_MJin / LHVBiom
)

# Gas blend - hydrogen:
# MJ_H2 -> kWh_H2 -> kg_H2 -> kWh electricity from photovoltaic system
CombinedCycle_GasBlend_Hydrogen_kWhH2 = (
    CombinedCycle_GasBlend_Hydrogen_MJin / 3.6
)

LHVH2_kWh_kg = (
    LHVH2 / 3.6
)

CombinedCycle_GasBlend_Hydrogen_kg = (
    CombinedCycle_GasBlend_Hydrogen_kWhH2
    / LHVH2_kWh_kg
)

# 55 kWh_el per kg_H2
CombinedCycle_GasBlend_Hydrogen_Input = (
    CombinedCycle_GasBlend_Hydrogen_kg * 55
)

# Standalone biomethane [m3]
CombinedCycle_Biomethane_Input = (
    CombinedCycle_Biomethane_MJin / LHVBiom
)

# Standalone hydrogen:
# MJ_H2 -> kWh_H2 -> kg_H2 -> kWh electricity from photovoltaic system
CombinedCycle_Hydrogen_kWhH2 = (
    CombinedCycle_Hydrogen_MJin / 3.6
)

CombinedCycle_Hydrogen_kg = (
    CombinedCycle_Hydrogen_kWhH2
    / LHVH2_kWh_kg
)

# 55 kWh_el per kg_H2
CombinedCycle_Hydrogen_Input = (
    CombinedCycle_Hydrogen_kg * 55
)

# ------------------------------------------------------------------
# Aggregated fuel energies for direct emissions [MJin, LHV]
# ------------------------------------------------------------------

# All non-H2 combustion-related flows
combined_cycle_non_hydrogen_energy_MJ = (
    CombinedCycle_GasBlend_NG_MJin
    + CombinedCycle_GasBlend_Biomethane_MJin
    + CombinedCycle_Biomethane_MJin
)

# Fossil-carbon combustion
combined_cycle_fossil_energy_MJ = (
    CombinedCycle_GasBlend_NG_MJin
)

# Biogenic-carbon combustion
combined_cycle_biogenic_energy_MJ = (
    CombinedCycle_GasBlend_Biomethane_MJin
    + CombinedCycle_Biomethane_MJin
)

# Total H2 input available for any future H2-specific direct-emission factors
combined_cycle_hydrogen_energy_MJ = (
    CombinedCycle_GasBlend_Hydrogen_MJin
    + CombinedCycle_Hydrogen_MJin
)

# ------------------------------------------------------------------
# Emission basis on HHV. The factors below are per-m3 values divided by HHV = 40.4 MJ/m3
# (e.g. CO2 0.05155 = 2.0826 kg/m3 / 40.4), i.e. per MJ fuel on an HHV basis, while the fuel
# energies above are on an LHV basis. Before, the factors were applied to LHV energy (-8.9 %).
# ------------------------------------------------------------------
_cc_ng_MJ_HHV = CombinedCycle_GasBlend_NG_MJin * (HHV / LHV)
_cc_biom_MJ_HHV = (CombinedCycle_GasBlend_Biomethane_MJin + CombinedCycle_Biomethane_MJin) * (HHVBiom / LHVBiom)
combined_cycle_non_hydrogen_energy_MJ_HHV = _cc_ng_MJ_HHV + _cc_biom_MJ_HHV
combined_cycle_fossil_energy_MJ_HHV = _cc_ng_MJ_HHV
combined_cycle_biogenic_energy_MJ_HHV = _cc_biom_MJ_HHV

# ------------------------------------------------------------------
# Direct emissions
# Factors are expressed per MJ of fuel input, HHV basis.
# ------------------------------------------------------------------

CombinedCycle_WaterCooling = (
    0.00662935643564359
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_WaterIn2 = (
    0.00579207920792081
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_WaterIn3 = (
    0.193069306930693
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_WaterOut4 = (
    9.65346534653468E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Acenaphthene = (
    7.65520E-13
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Acetaldehyde = (
    7.72277227722775E-10
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_AceticAcid = (
    1.1680693069307E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Arsenic = (
    3.96757425742575E-11
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Benzene = (
    8.70742574257428E-10
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Benzoapyrene = (
    5.10668316831685E-13
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Beryllium = (
    2.38440594059407E-12
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Butane = (
    8.93910891089111E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Cadmium = (
    2.18168316831684E-10
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_CarbonDioxideFossil = (
    0.0515495049504952
    * combined_cycle_fossil_energy_MJ_HHV
)

CombinedCycle_CarbonDioxideBiogenic = (
    0.0515495049504952
    * combined_cycle_biogenic_energy_MJ_HHV
)

CombinedCycle_CarbonMonoxideFossil = (
    2.12376237623763E-06
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Chromium = (
    2.77054455445546E-10
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Cobalt = (
    1.66039603960396E-11
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_DinitrogenMonoxide = (
    9.43143564356438E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Ethane = (
    1.32252475247525E-06
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Formaldehyde = (
    3.1180693069307E-08
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Hexane = (
    7.655198019802E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Lead = (
    9.94306930693072E-11
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Manganese = (
    7.53935643564359E-11
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Mercury = (
    6.64158415841586E-11
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_MethaneFossil = (
    9.37351485148517E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Nickel = (
    4.16064356435644E-10
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_NitrogenOxides = (
    2.46163366336634E-05
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_PAH = (
    7.72277227722775E-09
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Particulates = (
    4.73985148514853E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Pentane = (
    1.11014851485149E-06
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Propane = (
    6.80569306930695E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_PropionicAcid = (
    1.54455445544554E-08
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Selenium = (
    4.73985148514853E-12
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_SulfurDioxide = (
    5.52178217821784E-07
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_Toluene = (
    1.4480198019802E-09
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_WaterAir = (
    8.13864357863667E-05
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

CombinedCycle_WaterWater = (
    0.00674683138599583
    * combined_cycle_non_hydrogen_energy_MJ_HHV
)

## 3.6 Module C2

In [ ]:
CombinedCycle_Transport_to_treatment_plant = (50 * (mass/1000)) * AFQ_CombinedCycle

## 3.7 Module C3

In [ ]:
CombinedCycle_EOL_aluminiumCastalloy = CombinedCycle_aluminiumCastalloy
CombinedCycle_EOL_aluminiumWroughtalloy = CombinedCycle_aluminiumWroughtalloy
CombinedCycle_EOL_ceramictile = CombinedCycle_ceramictile
CombinedCycle_EOL_chromium = CombinedCycle_chromium
CombinedCycle_EOL_cobalt = CombinedCycle_cobalt
CombinedCycle_EOL_concreteAT = CombinedCycle_concreteAT
CombinedCycle_EOL_concreteCH = CombinedCycle_concreteCH
CombinedCycle_EOL_copper = CombinedCycle_copper
CombinedCycle_EOL_nickel = CombinedCycle_nickel
CombinedCycle_EOL_PEHD = CombinedCycle_PEHD
CombinedCycle_EOL_reinforcingsteel = CombinedCycle_reinforcingsteel
CombinedCycle_EOL_chromiumsteel = CombinedCycle_chromiumsteel
CombinedCycle_EOL_stonewool = CombinedCycle_stonewool

## 3.8 Module C4

In [ ]:
CombinedCycle_EOL_landfill_aluminium = (CombinedCycle_EOL_aluminiumCastalloy
                                        + CombinedCycle_EOL_aluminiumWroughtalloy) * LandfillMetals
CombinedCycle_EOL_landfill_ceramictile = CombinedCycle_EOL_ceramictile * LandfillClaybrick
CombinedCycle_EOL_landfill_chromium = CombinedCycle_EOL_chromium * LandfillMetals2
CombinedCycle_EOL_landfill_cobalt = CombinedCycle_EOL_cobalt * LandfillMetals2
CombinedCycle_EOL_landfill_concreteAT = CombinedCycle_EOL_concreteAT * LandfillConcrete
CombinedCycle_EOL_landfill_concreteCH = CombinedCycle_EOL_concreteCH * LandfillConcrete
CombinedCycle_EOL_landfill_copper = CombinedCycle_EOL_copper * LandfillMetals
CombinedCycle_EOL_landfill_nickel = CombinedCycle_EOL_nickel * LandfillMetals2
CombinedCycle_EOL_inc_PEHD = CombinedCycle_EOL_PEHD * IncinerationPEHD
CombinedCycle_EOL_landfill_PEHD = CombinedCycle_EOL_PEHD * LandfillPEHD
CombinedCycle_EOL_landfill_reinforcingsteel = CombinedCycle_EOL_reinforcingsteel * LandfillMetals
CombinedCycle_EOL_landfill_chromiumsteel = CombinedCycle_EOL_chromiumsteel * LandfillMetals2
CombinedCycle_EOL_landfill_stonewool = CombinedCycle_EOL_stonewool * LandfillMineralWool

## 3.9 Module D1 (related to the export of secondary materials)

In [ ]:
CombinedCycle_EOL_recycling_aluminiumCastalloy = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_aluminiumCastalloy) - (RecycledContentMetals * CombinedCycle_EOL_aluminiumCastalloy)
CombinedCycle_EOL_primary_aluminiumCastalloy = ((RecycledContentMetals * CombinedCycle_EOL_aluminiumCastalloy) 
                                                - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_aluminiumCastalloy)) * SRmetals

CombinedCycle_EOL_recycling_aluminiumWroughtalloy = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_aluminiumWroughtalloy) - (RecycledContentMetals * CombinedCycle_EOL_aluminiumWroughtalloy)
CombinedCycle_EOL_primary_aluminiumWroughtalloy = ((RecycledContentMetals * CombinedCycle_EOL_aluminiumWroughtalloy) - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_aluminiumWroughtalloy)) * SRmetals
CombinedCycle_EOL_recycling_chromium = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_chromium) - (RecycledContentMetals * CombinedCycle_EOL_chromium)
CombinedCycle_EOL_primary_chromium = ((RecycledContentMetals * CombinedCycle_EOL_chromium) - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_chromium)) * SRmetals
CombinedCycle_EOL_recycling_cobalt = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_cobalt) - (RecycledContentMetals * CombinedCycle_EOL_cobalt)
CombinedCycle_EOL_primary_cobalt = ((RecycledContentMetals * CombinedCycle_EOL_cobalt) - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_cobalt)) * SRmetals
CombinedCycle_EOL_recycling_concreteAT = (RecyclingConcrete * RecyclingEfficiencyConcrete * CombinedCycle_EOL_concreteAT) - (RecycledContentConcrete * CombinedCycle_EOL_concreteAT)
CombinedCycle_EOL_primary_concreteAT = ((RecycledContentConcrete * CombinedCycle_EOL_concreteAT) - (RecyclingConcrete * RecyclingEfficiencyConcrete * CombinedCycle_EOL_concreteAT)) * SRinert
CombinedCycle_EOL_recycling_concreteCH = (RecyclingConcrete * RecyclingEfficiencyConcrete * CombinedCycle_EOL_concreteCH) - (RecycledContentConcrete * CombinedCycle_EOL_concreteCH)
CombinedCycle_EOL_primary_concreteCH = ((RecycledContentConcrete * CombinedCycle_EOL_concreteCH) - (RecyclingConcrete * RecyclingEfficiencyConcrete * CombinedCycle_EOL_concreteCH)) * SRinert
CombinedCycle_EOL_recycling_copper = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_copper) - (RecycledContentMetals * CombinedCycle_EOL_copper)
CombinedCycle_EOL_primary_copper = ((RecycledContentMetals * CombinedCycle_EOL_copper) - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_copper)) * SRmetals
CombinedCycle_EOL_recycling_nickel = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_nickel) - (RecycledContentMetals * CombinedCycle_EOL_nickel)
CombinedCycle_EOL_primary_nickel = ((RecycledContentMetals * CombinedCycle_EOL_nickel) - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_nickel)) * SRmetals
CombinedCycle_EOL_recycling_HDPE = (RecyclingPEHD * RecyclingEfficiencyPlastics * CombinedCycle_EOL_PEHD) - (RecycledContentPlastic * CombinedCycle_EOL_PEHD)
CombinedCycle_EOL_primary_HDPE = ((RecycledContentPlastic * CombinedCycle_EOL_PEHD) - (RecyclingPEHD * RecyclingEfficiencyPlastics * CombinedCycle_EOL_PEHD)) * SRinert
CombinedCycle_EOL_recycling_reinforcingsteel = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_reinforcingsteel) - (RecycledContentMetals * CombinedCycle_EOL_reinforcingsteel)
CombinedCycle_EOL_primary_reinforcingsteel = ((RecycledContentMetals * CombinedCycle_EOL_reinforcingsteel) - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_reinforcingsteel)) * SRmetals
CombinedCycle_EOL_recycling_chromiumsteel = (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_chromiumsteel) - (RecycledContentMetals * CombinedCycle_EOL_chromiumsteel)
CombinedCycle_EOL_primary_chromiumsteel = ((RecycledContentMetals * CombinedCycle_EOL_chromiumsteel) - (RecyclingMetals * RecyclingEfficiencyMetals * CombinedCycle_EOL_chromiumsteel)) * SRmetals

## 3.10 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
CombinedCycle_EOL_energyrecoveryheat_PEHD = - CombinedCycle_EOL_inc_PEHD * LHVHDPE * Xincheat
CombinedCycle_EOL_energyrecoveryele_PEHD = (- CombinedCycle_EOL_inc_PEHD * LHVHDPE * Xincele) / 3.6

# 4. Data for export

In [ ]:
# Useful thermal energy produced by the Combined Cycle
# over the reference study period [kWh]
ThermalEnergyProduced_CombinedCycle = (
    np.asarray(ThermalEnergyproduced_primary, dtype=float)
    * RSP
)

# Electricity produced over the reference study period [kWh_el]
# (same W/Q ratio used in the allocation); used to charge it to its consumers
ElectricEnergyProduced_CombinedCycle = CombinedCycle_W_RSP